In [ ]:
import os
import sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import re
import gc
import time
import torch
import importlib
import importlib.metadata
import pandas as pd
import polars as pl
import io
import tqdm
import traceback
import json
import bitsandbytes
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

def apply_quant_patch():
    targets = [
        ("transformers.quantizers.quantizer_bnb_4bit", "Bnb4BitQuantizer"),
        ("transformers.quantizers.quantizer_bnb_4bit", "Bnb4bitQuantizer"),
        ("transformers.integrations.bitsandbytes", "Bnb4BitQuantizer"),
        ("transformers.modeling_utils", "Bnb4BitQuantizer")
    ]
    for mod_path, cls_name in targets:
        try:
            mod = importlib.import_module(mod_path)
            if hasattr(mod, cls_name):
                getattr(mod, cls_name).validate_environment = lambda *args, **kwargs: None
                return True
        except Exception: continue
    return False

if not apply_quant_patch():
    for attr in dir(transformers):
        if "Quantizer" in attr:
            q_cls = getattr(transformers, attr)
            if hasattr(q_cls, "validate_environment"):
                q_cls.validate_environment = lambda *args, **kwargs: None


In [ ]:
class HMADPipeline:
    def __init__(self, model_ids):
        self.model_ids = model_ids
        self.agents = {}
        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    def load_models(self):
        for role, path in self.model_ids.items():

            model = AutoModelForCausalLM.from_pretrained(
                path,
                quantization_config=self.bnb_config,
                device_map={"": 0}, # Force to the H100
                torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True, 
                trust_remote_code=True
            )
            
            tokenizer = AutoTokenizer.from_pretrained(path)
            tokenizer.padding_side = 'left'
            
            self.agents[role.lower()] = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer
            )

            del model
            gc.collect()
            torch.cuda.empty_cache()
            
        print("All agents loaded successfully.")

    def generate(self, agent_key, prompt, max_tokens=1024):
        formatted_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        
        outputs = self.agents[agent_key](
            formatted_prompt,
            max_new_tokens=max_tokens,
            temperature=0.1 if agent_key == "logician" else 0.5,
            do_sample=True,
            return_full_text=False,
            pad_token_id=self.agents[agent_key].tokenizer.eos_token_id
        )
        return outputs[0]['generated_text']

    def predict(self, problem):
        proposal = self.generate("logician", f"Solve step-by-step: {problem}")
        critique = self.generate("skeptic", f"Find errors in: {proposal}")
        code = self.generate("formalizer", f"Write Python code to verify the result of: {proposal}")
        
        final_prompt = (
            f"Problem: {problem}\nDraft: {proposal}\nCritique: {critique}\nCode: {code}\n"
            "Final corrected solution ending with 'The final answer is: \\boxed{number}'"
        )
        verdict = self.generate("logician", final_prompt)
        print(verdict)
        answer = self.normalize_latex(self.extract_final_answer(verdict))
        return answer

    def normalize_latex(self, text):
        text = re.sub(r'\s+','', text)
        return text

    def extract_final_answer(self, verdict):
        try:
            if not isinstance(verdict, str):
                return None
        
            verdict = verdict.replace("\x08", r"\b").replace("\x0c", r"\f")
            key = r"\boxed{"
            start = verdict.rfind(key)
            if start == -1:
                return None
        
            i = start + len(key)
            depth = 1
            out = []
            while i < len(verdict) and depth > 0:
                c = verdict[i]
                if c == "{":
                    depth += 1
                elif c == "}":
                    depth -= 1
                    if depth == 0:
                        break
                out.append(c)
                i += 1
            return "".join(out)
        except:
            return 0

In [ ]:
MODELS = {
    "logician": "/kaggle/input/models/qwen-lm/qwen2.5/transformers/32b-instruct/1",
    "formalizer": "/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2",
    "skeptic": "/kaggle/input/models/richolson/phi-3.5-mini-instruct/pytorch/default/1"
}

In [ ]:
hmad_engine = HMADPipeline(model_ids=MODELS)
hmad_engine.load_models()


In [ ]:
from torch.utils.data import Dataset

class ProblemDataset(Dataset):
    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

In [ ]:
from datasets import load_dataset

def download_data(n_per_type=20, seed=42):
    ds = load_dataset("AI-MO/NuminaMath-1.5", split="train", trust_remote_code=True)
    df = ds.to_pandas()
    df = df[df['source'].str.contains('amc', case=False, na=False)].copy()

    def is_simple_mc(text):
        if not isinstance(text, str): return False
        clean = re.sub(r'\\mathrm|\\text|[\(\)\{\}\s\$]', '', text)
        # Check if the result is strictly a single letter A, B, C, D, or E
        return len(clean) == 1 and clean.upper() in "ABCDE"

    df['is_mc'] = df['answer'].apply(is_simple_mc)
    mc_df = df[df['is_mc'] == True].drop(columns=['is_mc']).copy()
    mc_df = mc_df.dropna(subset=['answer', 'problem_type'])
    mc_df = mc_df[mc_df['problem_type'].isin(['Algebra','Geometry','Number Theory'])]
    mc_df['answer'] = mc_df['answer'].apply(lambda x: re.sub(r'\\mathrm|\\text|[\(\)\{\}\s\$]', '', x).upper())

    return mc_df

eval_data = download_data()
print(eval_data['problem_type'].value_counts())

In [ ]:
sampled = (eval_data.groupby("problem_type", group_keys=False).apply(lambda g: g.sample(n=20, random_state=42)).reset_index(drop=True))
print(sampled['problem_type'].value_counts())

In [ ]:
def extract_final_answer(self, text):
    if not isinstance(text, str) or not text:
        return None

    # 1.Convert actual Tab characters back to literal '\t'
    text = text.replace('\t', '\\t')

    # 2. Isolate the content of the LAST \boxed{}
    matches = re.findall(r'\\boxed\{(.+?)\}', text)
    if not matches:
        return None
    
    raw_content = matches[-1]

    # 3. Cleaning
    # Step A: Remove common LaTeX words even if the backslash is missing
    clean = re.sub(r'\\?text|\\?mathrm|\\?mathbf|\\?rm|\\?textbf', '', raw_content, flags=re.IGNORECASE)

    # Step B: Remove all remaining non-alphanumeric noise
    clean = re.sub(r'[^a-zA-Z0-9]', '', clean)

    # 4. Final choice match
    mc_match = re.search(r'[A-E]', clean, re.IGNORECASE)
    
    if mc_match:
        return mc_match.group(0).upper()
    
    num_match = re.search(r'\d+', clean)
    return num_match.group(0) if num_match else None

def clean_agent_output(text):
    """Strips chat headers and common LLM hallucinations to prevent input bloat."""
    if not isinstance(text, str): return ""
    tags = ["<|im_start|>", "<|im_end|>", "<|assistant|>", "<|user|>", "<|system|>", "assistant\n", "user\n"]
    for tag in tags:
        text = text.replace(tag, "")
    text = re.sub(r"Problem:.*?\n", "", text, flags=re.IGNORECASE)
    return text.strip()

def execute_code(code_str):
    import traceback, io, sys, re
    
    actual_code = code_str
    if "```python" in code_str.lower():
        try:
            blocks = re.findall(r"```python\n?(.*?)\n?```", code_str, re.DOTALL | re.IGNORECASE)
            if blocks:
                actual_code = blocks[-1].strip()
        except:
            actual_code = code_str

    output_capture = io.StringIO()
    sys.stdout = output_capture
    success = False
    
    try:
        exec_globals = {"__builtins__": __builtins__, "sp": sp, "symbols": symbols}
        exec(actual_code, exec_globals)
        raw_output = output_capture.getvalue()
        result = raw_output[:500] + ("..." if len(raw_output) > 500 else "")
        success = True
    except Exception:
        error_trace = traceback.format_exc()
        result = f"Execution Failed:\n{error_trace[-500:]}"
        success = False
    finally:
        sys.stdout = sys.__stdout__
    
    return result.strip() or "Success (No output)", success


In [ ]:
def format_for_logician(problems):
    """Wraps raw strings in the model's specific chat template."""
    formatted = []
    for p in problems:
        msg = [{"role": "user", "content": p}]
        # Using the logician's tokenizer as the standard for the pipeline
        f = hmad_engine.agents['logician'].tokenizer.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=True
        )
        formatted.append(f)
    return formatted

def _run(agent, dataset, batch_size, desc="Processing", max_new_tokens=512, **kwargs):
    """
    Standardized wrapper for pipeline execution with adaptive token limits.
    Now supports **kwargs to pass temperature, top_p, etc., to the model.
    """
    # Logic to handle Hugging Face's requirement for do_sample
    if 'temperature' in kwargs:
        temp = kwargs['temperature']
        if temp == 0:
            kwargs['do_sample'] = False
            kwargs.pop('temperature') 
        else:
            kwargs['do_sample'] = True
    results = list(tqdm.tqdm(
        agent(
            dataset, 
            batch_size=batch_size, 
            max_new_tokens=max_new_tokens, 
            return_full_text=False, 
            **kwargs  
        ), 
        total=len(dataset), 
        desc=desc
    ))
    
    return [clean_agent_output(out[0]['generated_text']) for out in results]

def build_final_prompt(problems, proposals, critiques, exec_results, level, original_drafts=None, code_audits=None):
    prompts = []
    for i in range(len(problems)):
        if level == 1:

            prompt = (
                f"Problem: {problems[i]}\n\n"
                "Instruction: Solve the problem above step-by-step. "
                "DO NOT repeat the problem text or these instructions in your response. "
                "Begin your solution immediately and be concise. "
                "The very last line of your output MUST be: 'The final answer is: \\boxed{LETTER}'"
            )
        else:
            draft_context = f"Initial Draft: {original_drafts[i]}\n" if original_drafts else ""
            audit_context = f"--- CODE AUDIT ---\n{code_audits[i]}\n\n" if code_audits and code_audits[i] != "No audit." else ""

            prompt = (
                f"--- PROBLEM ---\n{problems[i]}\n\n"
                f"--- DEBATE HISTORY ---\n{draft_context}Proposal: {proposals[i]}\nCritique: {critiques[i]}\n"
                f"--- CODE RESULT ---\n{exec_results[i]}\n\n"
                f"{audit_context}"
                "INSTRUCTION: You are the final judge. Review the debate and the code result.\n"
                "Note: If the Python code ran successfully, favor its numerical result over text-based logic.\n"
                "If the problem has multiple choice options (A, B, C, D, E), you MUST map the mathematical result to the correct option letter.\n"
                "Your very last line MUST be formatted exactly as: 'The final answer is: \\boxed{LETTER}' "
                "(where LETTER is A, B, C, D, or E). Do not put mathematical expressions in the box."
            )
        prompts.append(prompt)
    return prompts

def process_results(df, verdicts, proposals, critiques, codes, exec_results, level, elapsed_time):
    """Compiles detailed diagnostic data including the raw Python code generated."""
    debug_data, summary_results = [], []
    
    for i in range(len(df)):
        pred = hmad_engine.extract_final_answer(verdicts[i])
        raw_truth = f"\\boxed{{{df.iloc[i]['answer']}}}"
        truth = hmad_engine.extract_final_answer(raw_truth)
        is_correct = (str(pred).upper() == str(truth).upper()) if pred and truth else False
        
  
        debug_data.append({
            'level': level,
            'problem_type': df.iloc[i]['problem_type'],
            'problem': df.iloc[i]['problem'],
            'logician_proposal': proposals[i],
            'skeptic_critique': critiques[i],
            'formalizer_code': codes[i],      
            'code_result': exec_results[i],
            'raw_verdict': verdicts[i],
            'model_answer': pred,
            'ground_truth': truth,
            'is_correct': is_correct
        })
        
        summary_results.append({
            'problem_type': df.iloc[i]['problem_type'], 
            'is_correct': is_correct
        })

    # Summary Generation
    res_df = pd.DataFrame(summary_results)
    summary_df = res_df.groupby('problem_type')['is_correct'].agg(['count','sum','mean'])
    summary_df.columns = ['Total','Correct','Accuracy']
    
    overall = pd.DataFrame({
        'Total': [summary_df['Total'].sum()],
        'Correct': [summary_df['Correct'].sum()],
        'Accuracy': [summary_df['Correct'].sum() / summary_df['Total'].sum()]
    }, index=['OVERALL'])
    
    return pd.concat([summary_df, overall]), pd.DataFrame(debug_data)

In [ ]:
def run_benchmark(df, config_level=1, batch_size=10, log_file="benchmark_logs.jsonl"):
    """Runs the pipeline with deep telemetry, tiered temperatures, and batched healing."""
    
    if not os.path.exists(log_file):
        with open(log_file, "w", encoding="utf-8") as f:
            pass 

    all_debug_data = []
    print(f"Starting Deep Benchmark | Level: {config_level} | Batch: {batch_size}")

    for chunk_start in range(0, len(df), batch_size):
        chunk_end = min(chunk_start + batch_size, len(df))
        chunk_df = df.iloc[chunk_start:chunk_end].copy()
        problems = chunk_df['problem'].tolist()
        
        print(f"Processing Batch {chunk_start//batch_size + 1} (Problems {chunk_start} to {chunk_end-1})")
        phase_times = {}
        
        # --- PHASE 1: DRAFT (T=0.1) ---
        t0 = time.perf_counter()
        initial_draft_prompts = build_final_prompt(problems, [None]*len(problems), [None]*len(problems), [None]*len(problems), level=1)
        phase_1_proposals = _run(hmad_engine.agents['logician'], ProblemDataset(format_for_logician(initial_draft_prompts)), batch_size, temperature=0.1, desc="Phase 1: Drafting")
        phase_times['phase_1'] = time.perf_counter() - t0
        
        current_proposals = list(phase_1_proposals)
        accumulated_critiques = [""] * len(problems)
        code_audits = ["No audit."] * len(problems)
        codes, exec_results = [""]*len(problems), [""]*len(problems)
        first_try_success, final_code_success = [False]*len(problems), [False]*len(problems)
        debate_time_total = 0.0

        # --- PHASE 2 & 3: Debate (Skeptic T=0.7, Logician T=0.2) ---
        if config_level >= 2:
            rounds = min(config_level - 1, 2) 
            for r in range(rounds):
                t0_round = time.perf_counter()
                crit_in = [f"Prob: {p}\nDraft: {pr}\nFind errors." for p, pr in zip(problems, current_proposals)]
                round_critiques = _run(hmad_engine.agents['skeptic'], ProblemDataset(crit_in), batch_size, temperature=0.7, desc=f"Phase 2: Skeptic (R{r+1})")
                
                for i in range(len(problems)):
                    accumulated_critiques[i] += f"\n[Round {r+1}]: {round_critiques[i]}"

                ref_in = [(f"Problem: {p}\nDraft: {pr}\nCritique: {c}\n\n"
                           "Review the critique. The skeptic might be wrong. If real error, fix it. "
                           "If hallucination, defend and maintain.") for p, pr, c in zip(problems, current_proposals, round_critiques)]
                current_proposals = _run(hmad_engine.agents['logician'], ProblemDataset(ref_in), batch_size, temperature=0.2, desc=f"Phase 3: Refining (R{r+1})")
                debate_time_total += (time.perf_counter() - t0_round)
            phase_times['phase_2_3'] = debate_time_total

        # --- PHASE 4:Coding & Self Correction ---
        if config_level >= 2:
            t0 = time.perf_counter()
            form_in = [f"Prob: {p}\nLogic:\n{pr}\n\n```python\n" for p, pr in zip(problems, current_proposals)]
            codes = _run(hmad_engine.agents['formalizer'], ProblemDataset(form_in), batch_size, temperature=0.1, desc="Phase 4: Coding")
            
            initial_outputs = [execute_code(c) for c in codes]
            exec_results = [o[0] for o in initial_outputs]
            first_try_success = [o[1] for o in initial_outputs]
            final_code_success = list(first_try_success)
            
            failed_idx = [idx for idx, ok in enumerate(first_try_success) if not ok]
            if failed_idx:
                print(f"🛠️  Healing {len(failed_idx)} failed scripts...")
                retry_p = [f"Prob: {problems[i]}\nCode failed: {exec_results[i]}\nFix it:\n```python\n" for i in failed_idx]
                healed = _run(hmad_engine.agents['formalizer'], ProblemDataset(retry_p), batch_size, temperature=0.5, desc="Phase 4.5: Healing")
                
                for r_idx, o_idx in enumerate(failed_idx):
                    codes[o_idx] = healed[r_idx]
                    res, ok = execute_code(healed[r_idx])
                    exec_results[o_idx], final_code_success[o_idx] = res, ok
            phase_times['phase_4_heal'] = time.perf_counter() - t0

        # --- PHASE 5: Final synthesis (T=0.0) ---
        t0 = time.perf_counter()
        if config_level >= 2:
            verdict_in = build_final_prompt(problems, current_proposals, accumulated_critiques, exec_results, config_level, 
                                            original_drafts=phase_1_proposals, code_audits=code_audits)
            verdicts = _run(hmad_engine.agents['logician'], ProblemDataset(verdict_in), batch_size, temperature=0.0, desc="Phase 5: Synthesis")
        else:
            verdicts = phase_1_proposals
        phase_times['phase_final'] = time.perf_counter() - t0

        # --- Metric Calculation ---
        def clean_mcq_answer(ans):
            if not ans: return ""
            ans_str = str(ans).strip().upper()
            match = re.search(r'\\text\{([^}]+)\}', ans_str)
            return match.group(1).strip() if match else ans_str

        for i in range(len(problems)):
            ground_truth = clean_mcq_answer(chunk_df.iloc[i]['answer'])
            final_ans = clean_mcq_answer(hmad_engine.extract_final_answer(verdicts[i]))
            p1_ans = clean_mcq_answer(hmad_engine.extract_final_answer(phase_1_proposals[i]))
            is_correct = (final_ans == ground_truth) if final_ans and ground_truth else False
            p1_correct = (p1_ans == ground_truth) if p1_ans and ground_truth else False
            
            row_data = {
                "problem_type": chunk_df.iloc[i]['problem_type'],
                "problem": problems[i],
                "ground_truth": ground_truth,
                "phase_1_answer": p1_ans,
                "final_answer": final_ans,
                "is_correct": is_correct,
                "phase_1_correct": p1_correct,
                "positive_flip": (not p1_correct) and is_correct,
                "negative_flip": p1_correct and (not is_correct),
                "first_try_success": first_try_success[i],
                "final_code_success": final_code_success[i],
                "self_healed": (not first_try_success[i]) and final_code_success[i],
                "code_result_raw": exec_results[i],
                "word_count_phase_1": len(phase_1_proposals[i].split()),
                "word_count_code": len(codes[i].split()),
                "word_count_verdict": len(verdicts[i].split()),
                "time_phase_1": phase_times.get('phase_1', 0) / len(problems),
                "time_phase_4_code": phase_times.get('phase_4_heal', 0) / len(problems),
                "time_phase_final": phase_times.get('phase_final', 0) / len(problems),
                "raw_phase_1": phase_1_proposals[i],
                "raw_critique": accumulated_critiques[i],
                "raw_code": codes[i],
                "raw_verdict": verdicts[i],
            }
            
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(row_data) + "\n")
            all_debug_data.append(row_data)

    # ---Summary ---
    debug_df = pd.DataFrame(all_debug_data)
    summary_df = debug_df.groupby('problem_type')['is_correct'].agg(['count','sum','mean'])
    summary_df.columns = ['Total','Correct','Accuracy']
    
    overall = pd.DataFrame({
        'Total': [summary_df['Total'].sum()],
        'Correct': [summary_df['Correct'].sum()],
        'Accuracy': [summary_df['Correct'].sum() / summary_df['Total'].sum()] if summary_df['Total'].sum() > 0 else [0]
    }, index=['OVERALL'])
    
    return pd.concat([summary_df, overall]), debug_df

In [ ]:
#Execution
summary_baseline, debug_baseline = run_benchmark(
    df=sampled,           
    config_level=2,        
    batch_size=25, 
    log_file="final_update_60_config2.jsonl" 
)